# 🌀 CHFT v2 — Campos Holográficos de Fourier: Benchmark Completo

**Paradigma**: Representaciones FHRR (Fourier Holographic Reduced Representation) + Memoria de Hopfield Moderna.

Este es un modelo **no-LLM, no-Transformer**: aprende mediante ajuste de fases complejas en el círculo unitario.

### Métricas evaluadas:
| Métrica | Descripción |
|---|---|
| `Loss` | Cross-Entropy por época (train + val) |
| `Accuracy@1` | % de tokens exactos predichos en validación |
| `Perplexity` | Exponencial del loss medio (menor = mejor) |
| `Diversity` | % tokens únicos en texto generado (evita repetición) |

> **Runtime recomendado**: GPU T4 → Runtime → Change runtime type → T4 GPU

In [ ]:
# Instalar dependencias
!pip install -q datasets tiktoken

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import math
import collections
from datasets import load_dataset
import tiktoken
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ─────────────────────────────────────────────────────────────
# CONFIGURACIÓN — optimizada para GPU T4 de Google Colab
# ─────────────────────────────────────────────────────────────
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DIMENSION     = 4096    # Mayor dimensión aprovecha VRAM de la T4
CONTEXT_LEN   = 8       # Ventana de contexto
EPOCHS        = 10      # Épocas con scheduler coseno
LEARNING_RATE = 0.01
BATCH_SIZE    = 1024    # Batch grande para GPU
NUM_STORIES   = 1000    # 5x más que versión CPU
TOPK          = 5       # Top-k para sampling diverso
VAL_SPLIT     = 0.1     # 10% para validación

print(f'Dispositivo : {DEVICE}')
print(f'Dimension   : {DIMENSION:,}')
print(f'Stories     : {NUM_STORIES}')
print(f'Epocas      : {EPOCHS}')

## 1. 📥 Dataset & Tokenización

In [ ]:
print('[1/5] Cargando TinyStories...')
dataset   = load_dataset('roneneldan/TinyStories', split=f'train[:{NUM_STORIES}]')
tokenizer = tiktoken.get_encoding('cl100k_base')

all_token_ids     = set()
stories_tokenized = []

for item in dataset:
    tokens = tokenizer.encode(item['text'])
    stories_tokenized.append(tokens)
    all_token_ids.update(tokens)

vocab        = list(all_token_ids)
vocab_size   = len(vocab)
token_to_idx = {tok: i for i, tok in enumerate(vocab)}
idx_to_token = {i: tok for i, tok in enumerate(vocab)}

print(f'Vocabulario : {vocab_size:,} tokens unicos')

# Construir pares (contexto, objetivo)
contexts_list, targets_list = [], []
for story in stories_tokenized:
    if len(story) < CONTEXT_LEN + 1:
        continue
    idx_seq = [token_to_idx[t] for t in story]
    for i in range(len(idx_seq) - CONTEXT_LEN):
        contexts_list.append(idx_seq[i : i + CONTEXT_LEN])
        targets_list.append(idx_seq[i + CONTEXT_LEN])

# Shuffle y split train/val
num_total = len(contexts_list)
num_val   = int(num_total * VAL_SPLIT)
num_train = num_total - num_val

ctx_t = torch.tensor(contexts_list, dtype=torch.long)
tgt_t = torch.tensor(targets_list,  dtype=torch.long)
perm  = torch.randperm(num_total)
ctx_t, tgt_t = ctx_t[perm].to(DEVICE), tgt_t[perm].to(DEVICE)

train_ctx, val_ctx = ctx_t[:num_train], ctx_t[num_train:]
train_tgt, val_tgt = tgt_t[:num_train], tgt_t[num_train:]

print(f'Train       : {num_train:,} muestras')
print(f'Validacion  : {num_val:,} muestras')

## 2. 🧠 Arquitectura CHFT: FHRR + Hopfield Moderno

**Idea central**: cada token = fasor complejo `e^{iθ}` en el círculo unitario.
El contexto = superposición de fasores. El modelo aprende ajustando las fases θ.

In [ ]:
class FHRRPhasorEmbedding(nn.Module):
    """
    Fourier Holographic Reduced Representation.
    Cada token -> vector de fases theta en [-pi, pi]^D
    Hipervector complejo: e^{i*theta} = cos(theta) + i*sin(theta)
    """
    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        self.phases = nn.Parameter(
            torch.empty(num_embeddings, embedding_dim).uniform_(-math.pi, math.pi)
        )

    def forward(self, indices: torch.Tensor) -> torch.Tensor:
        phi = self.phases[indices]
        return torch.complex(torch.cos(phi), torch.sin(phi))

    def all_keys(self) -> torch.Tensor:
        return torch.complex(torch.cos(self.phases), torch.sin(self.phases))


class ModernHopfieldMemory:
    """
    Recuperacion de atractores por interferencia constructiva.
    Equivalente a cross-attention sin parametros adicionales.
    """
    def __init__(self, beta: float = 16.0):
        self.beta = beta
        self.keys = None

    @torch.no_grad()
    def update_keys(self, codebook):
        self.keys = codebook.all_keys().detach()

    @torch.no_grad()
    def query_topk(self, state: torch.Tensor) -> torch.Tensor:
        """Retorna logits de similitud para todos los tokens del vocabulario"""
        return torch.real(
            torch.matmul(self.keys, torch.conj(state).unsqueeze(-1))
        ).squeeze(-1)


codebook    = FHRRPhasorEmbedding(vocab_size, DIMENSION).to(DEVICE)
hopfield    = ModernHopfieldMemory(beta=16.0)
hopfield.update_keys(codebook)

params = sum(p.numel() for p in codebook.parameters())
print(f'Parametros: {params:,}  ({params * 4 / 1e6:.1f} MB en float32)')
print(f'Modelo CHFT inicializado en {DEVICE}')

## 3. 🏋️ Entrenamiento con Train / Val

In [ ]:
print('[2/5] Iniciando Entrenamiento CHFT...')
optimizer = torch.optim.Adam(codebook.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

loss_history     = []
val_loss_history = []
t0 = time.time()

for epoch in range(EPOCHS):
    # ── TRAIN ──
    codebook.train()
    epoch_loss  = 0.0
    perm        = torch.randperm(num_train)
    num_batches = math.ceil(num_train / BATCH_SIZE)

    for b in range(num_batches):
        idx   = perm[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        ctx_b = train_ctx[idx]
        tgt_b = train_tgt[idx]

        # Superposicion holografica: psi = sum(e^{i*theta_t})
        ctx_hv   = codebook(ctx_b)                               # [B, C, D]
        psi      = torch.sum(ctx_hv, dim=1)                     # [B, D]
        psi      = nn.functional.normalize(psi, p=2, dim=1)     # normalizar
        keys_all = codebook.all_keys()                           # [V, D]
        logits   = torch.real(torch.matmul(psi, torch.conj(keys_all).t()))  # [B, V]

        loss = nn.functional.cross_entropy(logits * 6.0, tgt_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            codebook.phases.copy_(
                torch.remainder(codebook.phases + math.pi, 2 * math.pi) - math.pi
            )
        epoch_loss += loss.item()

    scheduler.step()
    hopfield.update_keys(codebook)

    # ── VALIDACION ──
    codebook.eval()
    val_loss_sum = 0.0
    val_batches  = math.ceil(num_val / BATCH_SIZE)
    with torch.no_grad():
        for b in range(val_batches):
            ctx_v    = val_ctx[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
            tgt_v    = val_tgt[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
            ctx_hv   = codebook(ctx_v)
            psi_v    = nn.functional.normalize(torch.sum(ctx_hv, dim=1), p=2, dim=1)
            keys_all = codebook.all_keys()
            logits_v = torch.real(torch.matmul(psi_v, torch.conj(keys_all).t()))
            val_loss_sum += nn.functional.cross_entropy(logits_v * 6.0, tgt_v).item()

    avg_train = epoch_loss / num_batches
    avg_val   = val_loss_sum / val_batches
    loss_history.append(avg_train)
    val_loss_history.append(avg_val)
    print(f'  Epoch {epoch+1:02d}/{EPOCHS} | Train: {avg_train:.4f} | Val: {avg_val:.4f}')

elapsed = time.time() - t0
print(f'\nEntrenamiento completado en {elapsed:.1f}s ({elapsed/60:.1f} min)')

## 4. 📊 Benchmark — Métricas Cuantitativas

In [ ]:
print('[3/5] Calculando metricas...')
codebook.eval()

# ── Accuracy@1 y Perplexity ──
correct  = 0
total_ce = 0.0
n_val    = 0

with torch.no_grad():
    for b in range(math.ceil(num_val / BATCH_SIZE)):
        ctx_v    = val_ctx[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        tgt_v    = val_tgt[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]
        ctx_hv   = codebook(ctx_v)
        psi_v    = nn.functional.normalize(torch.sum(ctx_hv, dim=1), p=2, dim=1)
        keys_all = codebook.all_keys()
        logits_v = torch.real(torch.matmul(psi_v, torch.conj(keys_all).t())) * 6.0
        preds    = torch.argmax(logits_v, dim=1)
        correct += (preds == tgt_v).sum().item()
        total_ce += nn.functional.cross_entropy(logits_v, tgt_v, reduction='sum').item()
        n_val    += len(tgt_v)

accuracy   = correct / n_val * 100
perplexity = math.exp(total_ce / n_val)

# ── Baseline: siempre predice el token mas frecuente ──
freq_counter = collections.Counter(train_tgt.cpu().tolist())
most_common  = freq_counter.most_common(1)[0][0]
base_correct = sum(1 for t in val_tgt.cpu().tolist() if t == most_common)
base_acc     = base_correct / num_val * 100

print(f'\n  === RESULTADOS ===')
print(f'  Accuracy@1  (CHFT)     : {accuracy:.2f}%')
print(f'  Accuracy@1  (baseline) : {base_acc:.2f}%')
print(f'  Ganancia vs baseline   : +{accuracy - base_acc:.2f} puntos porcentuales')
print(f'  Perplexity             : {perplexity:.2f}  (azar = {vocab_size:,})')

## 5. ✍️ Generación con Top-k Sampling

Se usa **top-k sampling con temperatura** para evitar la repetición que produce el argmax puro.

In [ ]:
def generate(prompt: str, max_new: int = 25, k: int = TOPK, temp: float = 0.8) -> str:
    codebook.eval()
    tokens_in = tokenizer.encode(prompt)
    valid_tok = [t for t in tokens_in if t in token_to_idx]
    if not valid_tok:
        return '[tokens fuera del vocabulario]'
    gen_idx = [token_to_idx[t] for t in valid_tok]

    with torch.no_grad():
        for _ in range(max_new):
            ctx = gen_idx[-CONTEXT_LEN:]
            if len(ctx) < CONTEXT_LEN:
                ctx = ctx + [ctx[-1]] * (CONTEXT_LEN - len(ctx))
            ctx_t  = torch.tensor(ctx, device=DEVICE)
            ctx_hv = codebook(ctx_t)
            psi    = nn.functional.normalize(torch.sum(ctx_hv, dim=0), p=2, dim=0)
            logits = hopfield.query_topk(psi)
            # Top-k + temperatura
            topk_vals, topk_ids = torch.topk(logits, k)
            probs  = torch.softmax(topk_vals / temp, dim=0)
            chosen = topk_ids[torch.multinomial(probs, 1).item()].item()
            gen_idx.append(chosen)

    return tokenizer.decode([idx_to_token[i] for i in gen_idx])


print('[4/5] Generacion de texto (Top-k={}, Temperatura=0.8):\n'.format(TOPK))

test_prompts = [
    ('Once upon a time',      25),
    ('A little girl saw',     25),
    ('The cat went to',       25),
    ('There was a small dog', 25),
    ('The sun was shining',   25),
]

for prompt, length in test_prompts:
    out = generate(prompt, max_new=length)
    print(f'Prompt : {prompt}')
    print(f'Output : {out}\n')

# ── Diversity score ──
all_gen_tokens = []
for prompt, _ in test_prompts:
    all_gen_tokens.extend(tokenizer.encode(generate(prompt, max_new=50)))

diversity = len(set(all_gen_tokens)) / len(all_gen_tokens) * 100
print(f'Diversity score : {diversity:.1f}%  (tokens unicos / tokens totales generados)')

## 6. 📈 Visualización Completa de Resultados

In [ ]:
print('[5/5] Graficando...')
epochs_ax = range(1, EPOCHS + 1)

fig = plt.figure(figsize=(14, 10))
fig.suptitle('CHFT v2 — Resultados del Benchmark', fontsize=15, fontweight='bold', y=0.98)
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.35)

# Panel 1: Curvas de perdida
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(epochs_ax, loss_history,     'o-', color='#7C3AED', lw=2, label='Train Loss')
ax1.plot(epochs_ax, val_loss_history, 's--', color='#EC4899', lw=2, label='Val Loss')
ax1.fill_between(epochs_ax, loss_history, val_loss_history, alpha=0.08, color='gray')
ax1.set_title('Curva de Perdida (Cross-Entropy)')
ax1.set_xlabel('Epoca')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xticks(list(epochs_ax))

# Panel 2: Accuracy comparativa
ax2 = fig.add_subplot(gs[1, 0])
labels = ['Baseline\n(frecuencia)', 'CHFT v2\n(nuestro)']
values = [base_acc, accuracy]
colors = ['#94A3B8', '#7C3AED']
bars   = ax2.bar(labels, values, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
             f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax2.set_title('Accuracy@1: CHFT vs Baseline')
ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim(0, max(values) * 1.35)
ax2.grid(True, axis='y', alpha=0.3)

# Panel 3: Tabla de metricas
ax3 = fig.add_subplot(gs[1, 1])
ax3.axis('off')
rows = [
    ['Vocabulario',      f'{vocab_size:,} tokens'],
    ['Historias',        f'{NUM_STORIES:,}'],
    ['Dim. FHRR',        f'{DIMENSION:,}'],
    ['Muestras train',   f'{num_train:,}'],
    ['Epocas',           str(EPOCHS)],
    ['Tiempo total',     f'{elapsed:.0f}s'],
    ['Train Loss final', f'{loss_history[-1]:.4f}'],
    ['Val Loss final',   f'{val_loss_history[-1]:.4f}'],
    ['Accuracy@1',       f'{accuracy:.2f}%'],
    ['Perplexity',       f'{perplexity:.1f}'],
    ['Diversity',        f'{diversity:.1f}%'],
]
tbl = ax3.table(cellText=rows, colLabels=['Parametro', 'Valor'],
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.45)
ax3.set_title('Resumen de Metricas', pad=12)

plt.savefig('chft_benchmark_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada como chft_benchmark_results.png')

## 7. 💾 Guardar Modelo en Google Drive

In [ ]:
from google.colab import drive
import os, pickle, shutil

drive.mount('/content/drive')

dest_dir = '/content/drive/MyDrive/colabStore/01-CHFT'
os.makedirs(dest_dir, exist_ok=True)

model_data = {
    'phases':       codebook.phases.cpu().detach(),
    'token_to_idx': token_to_idx,
    'idx_to_token': idx_to_token,
    'vocab':        vocab,
    'vocab_size':   vocab_size,
    'dimension':    DIMENSION,
    'context_len':  CONTEXT_LEN,
    'metrics': {
        'accuracy_pct':     accuracy,
        'perplexity':       perplexity,
        'baseline_acc_pct': base_acc,
        'diversity_pct':    diversity,
        'train_loss_final': loss_history[-1],
        'val_loss_final':   val_loss_history[-1],
        'epochs':           EPOCHS,
        'training_secs':    elapsed,
    }
}

model_path = os.path.join(dest_dir, 'chft_model_v2.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

shutil.copy('chft_benchmark_results.png', os.path.join(dest_dir, 'chft_benchmark_results.png'))

print(f'Modelo guardado : {model_path}')
print(f'Grafico guardado: {dest_dir}/chft_benchmark_results.png')
print(f'\n=== RESUMEN FINAL ===')
print(f'  Accuracy@1  : {accuracy:.2f}%')
print(f'  Perplexity  : {perplexity:.2f}')
print(f'  Diversity   : {diversity:.1f}%')
print(f'  Tiempo      : {elapsed:.0f}s')